In [45]:
from dotenv import load_dotenv
import os
import requests
from typing import *
import sys
import subprocess
import shlex
from datetime import datetime
import json
import uuid
import pandas as pd
from google.cloud import storage

# Add path to import custom modules
# sys.path.append(os.path.abspath("../src"))

load_dotenv()

True

In [46]:
# logger_config.py
import logging

def configure_logging():
    # Configure the root logger
    root_logger = logging.getLogger()
    
    # Check if handlers already exist to avoid duplicates
    if root_logger.handlers:
        return
        
    # Create a stream handler for console output
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    
    # Create formatter and add to the handler
    formatter = logging.Formatter('[%(asctime)s] %(name)s:%(lineno)d - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    
    # Add the handler to the root logger
    root_logger.addHandler(console_handler)
    root_logger.setLevel(logging.INFO)

def get_module_logger(module_name):
    # Make sure logging is configured
    configure_logging()
    
    # Return a logger with the module name
    return logging.getLogger(module_name)

In [47]:
# Auxilary functions
def partition_id_by_year_quarter(p):
    return "".join(p.get('display_name').split(" ")[:2])

def partition_id_by_year(p):
    return p.get('display_name').split(" ")[0]

def no_of_parts_in_partition(p):
    return int(p.get('display_name').replace("(", "").replace(")","").split(" ")[-1])

def part_size_mb(p):
    return float(p.get('size_mb'))

def get_total_size(json):
    return round(sum([p.get('size_mb') for p in json.get('partitions')]),2)

def read_json_file(json_path):
    with open(json_path, "r") as f:
        d = json.load(f)
        return d
        
def get_filename(url):
    return ".".join("_".join(url.split("/")[-2:]).split(".")[:-1])

def download_file(url, download_path="tmp"):
    filename = get_filename(url)
    os.makedirs(download_path,exist_ok=True)
    subprocess.run(
        f'wget -q -O - {url} | gunzip > {os.path.join(download_path,filename)}',
        shell=True,
    )

def filter_partition(years='', partitions=[]):
    years = [y.strip() for y in years.split(",")]
    if not years:
        print("No args provided")
        return
    return [p for p in partitions if p.get('partition_id') in years]

In [48]:
# Extract_drug_events
"""
Function to restructure JSON object to handle batch processing better
"""
def extract_drug_events(data):
    events = data.get('results').get('drug').get('event')
    total_records = events.get('total_records')
    partitions = events.get('partitions')

    # Generate unique partition_id and count set
    partition_ids = {}
    for p in partitions:
        id = partition_id_by_year(p)
        partition_ids[id] = partition_ids.get(id,0) + 1
    
    # Groups partition by partitionid
    results = []
    for item in partition_ids.items():
        id, count = item
        file_list = []
        counter = 0
        tot_size = 0

        for p in partitions:
            if counter == count:
                break
            if partition_id_by_year(p) == id:
                counter+=1
                file_list.append(p.get('file'))
                tot_size+=part_size_mb(p)
                
        results.append(
            {
                "partition_id": id,
                "count": count,
                "size_mb" : round(tot_size,2),
                "files" : file_list
            }
        )
    
    return {
        "total_records" : total_records,
        "partitions" : results
    }

In [49]:
# Create Batch
"""
Function to seggregate partitions as batches based on disksize threshold
"""
def create_batch(partitions, max_batch_size_mb=10000):
    batch = []                  # partitions per batch
    batch_partitions = []       # Partitions under the threshold
    big_batch_partitions = []   # Different approach to process bigger partitions
    sum_size = 0                # Size counter

    for p in partitions:
        size = p.get('size_mb', 0)

        if size > max_batch_size_mb:
            # TODO:
            # Handle oversized partititions
            big_batch_partitions.append(p)
            continue
        
        if sum_size + size > max_batch_size_mb:
            # TODO:
            # - Declare batch_partitions as batch #
            # - Reset sum_size
            # - Reset batch_partitions
            batch.append(batch_partitions.copy())
            batch_partitions.clear()
            sum_size = 0
            continue
        
        batch_partitions.append(p)
        sum_size += size

    # Flush batch_partitions to schedule as last batch
    if len(batch_partitions) != 0:
        batch.append(batch_partitions.copy())
        batch_partitions.clear()
    
    return batch, big_batch_partitions

In [50]:
# ADE class
logger = get_module_logger(__name__)

class ADE:
    schema = ['patients', 'drugs', 'reactions']

    # Patient information
    patient_header = [
        "patientid",
        "patientagegroup",
        "patientonsetage",
        "patientonsetageunit",
        "patientsex",
        "patientweight",
        "fulfillexpeditecriteria",                           
        "primarysourcecountry",                              
        "occurcountry",                                      
        "reporttype",                                        
        "receiptdate",
        "receivedate",
        "safetyreportid",
        "transmissiondate",                                  
        "serious",
        "seriousnesscongenitalanomali",                      
        "seriousnessdeath",
        "seriousnesshospitalization",
        "seriousnessdisabling",
        "seriousnesslifethreatening",
        "seriousnessother",
    ]

    # Drug information
    drug_header = [
        "patientid",
        "actiondrug",                                     
        "drugcharacterization",                           
        "medicinalproduct",
        "activesubstancename",
        "drugindication",    
        "drugadministrationroute",    
        "drugstartdate",
        "drugenddate",
        "drugdosagetext",
        "drugstructuredosagenumb",
        "drugstructuredosageunit",
        "drugtreatmentduration",
        "drugtreatmentdurationunit",
        "drugrecurreadministration",
    ]

    # Reaction information
    reaction_header = [
        "patientid",
        "reactionmeddrapt",
        "reactionoutcome",
    ]

    def __init__(self):
        # Initialize Prometheus here

        self.patients_list = []
        self.drugs_list = []
        self.reactions_list = []
    
    def extractJSON(self, json):
        data = json.get('results')
        for item in data:
            patientid = str(uuid.uuid4())
            patient = item.get("patient",{})

            self.patients_list.append((
                patientid,
                patient.get("patientagegroup"),
                patient.get("patientonsetage"),
                patient.get("patientonsetageunit"),
                patient.get("patientsex"),
                patient.get("patientweight"),
                item.get("fulfillexpeditecriteria"),                            # Added
                item.get("primarysourcecountry"),                               # Added
                item.get("occurcountry"),                                       # Added
                item.get("reporttype"),                                         # Added
                item.get("receiptdate"),
                item.get("receivedate"),
                item.get("safetyreportid"),
                item.get("transmissiondate"),                                   # Added
                item.get("serious"),
                item.get("seriousnesscongenitalanomali"),                       # Added
                item.get("seriousnessdeath"),
                item.get("seriousnesshospitalization"),
                item.get("seriousnessdisabling"),
                item.get("seriousnesslifethreatening"),
                item.get("seriousnessother"),
            ))

            drugs = patient.get('drug',[])
            for drug in drugs:
                self.drugs_list.append((
                    patientid,
                    drug.get("actiondrug"),                                     # Added
                    drug.get("drugcharacterization"),                           # Added

                    drug.get("medicinalproduct"),
                    drug.get("activesubstance",{}).get("activesubstancename"),
                    drug.get("drugindication"),    
                    drug.get("drugadministrationroute"),    
                    drug.get("drugstartdate"),
                    drug.get("drugenddate"),
                    drug.get("drugdosagetext"),
                    drug.get("drugstructuredosagenumb"),
                    drug.get("drugstructuredosageunit"),
                    drug.get("drugtreatmentduration"),
                    drug.get("drugtreatmentdurationunit"),
                    drug.get("drugrecurreadministration"),
                ))

            reactions = patient.get("reaction",[])
            for reaction in reactions:
                self.reactions_list.append((
                    patientid,
                    reaction.get("reactionmeddrapt"),
                    reaction.get("reactionoutcome"),
                ))

    def _to_dataframe(self):
        df_patients = pd.DataFrame(self.patients_list, columns=self.patient_header)
        df_drugs = pd.DataFrame(self.drugs_list, columns=self.drug_header)
        df_reactions = pd.DataFrame(self.reactions_list, columns=self.reaction_header)

        return df_patients, df_drugs, df_reactions

    def save_as_parquet(self, save_to, fname, subfolder):
        df_patients, df_drugs, df_reactions = self._to_dataframe()
        df = [df_patients, df_drugs, df_reactions]

        dirs = []

        for p in ["patient", "drug", "reaction"]:
            path = os.path.join(save_to, p, subfolder)
            dirs.append(path)
            if not os.path.exists(path):
                logger.info(f"Directory '{path}' missing. Created '{path}'")
                os.makedirs(path, exist_ok=True)
        
        for d,p in zip(df, dirs):
            saved_path = os.path.join(p,f"{fname}.parquet")
            d.to_parquet(saved_path)
            logger.info(f"Parquet File saved to: {saved_path}")

In [51]:
# Link to downloads of Drug Event data
res = requests.get("https://api.fda.gov/download.json")
data = res.json()
downloads_json = extract_drug_events(data)

partition = filter_partition(years='2004', partitions=downloads_json.get('partitions'))
partition

[{'partition_id': '2004',
  'count': 20,
  'size_mb': 1040.06,
  'files': ['https://download.open.fda.gov/drug/event/2004q3/drug-event-0001-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0002-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0003-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0004-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q3/drug-event-0005-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0001-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0002-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0003-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0004-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q2/drug-event-0005-of-0005.json.zip',
   'https://download.open.fda.gov/drug/event/2004q1/drug-e

In [52]:
# # Download partition's files
# for i in partition[0].get('files'):
#     download_file(i)

In [53]:
def null_ratio(records, null_counts):
    schema = ['patients', 'drugs', 'reactions']
    null_ratio = {k:{} for k in schema}

    for s in schema:
        for k,v in null_counts[s].items():
            null_ratio[s][k] = float(v / records[s])

    return null_ratio

In [56]:
# from pprint import pprint

# schema = ['patients', 'drugs', 'reactions']
# total_records = {k:0 for k in schema}
# total_null_count = {k:{} for k in schema}

# # Simulating a partition
# for f in os.listdir('tmp'):
#     filepath = os.path.join('tmp',f)
#     file_json = read_json_file(filepath)

#     ade = ADE()
#     ade.extractJSON(file_json)

#     # Metrics
#     patients_df, drugs_df, reactions_df = ade._to_dataframe()
#     dfs = [patients_df, drugs_df, reactions_df]

#     for s, df in zip(schema, dfs):
#         # Calculate total records
#         total_records[s] = total_records[s] + df.shape[0]

#         # Calculate null counts
#         null_count = df.isna().sum().to_dict()
#         for k, v in null_count.items():
#             total_null_count[s][k] = total_null_count[s].get(k, 0) + v

# null_ratio_results = null_ratio(total_records, total_null_count)

In [57]:
# pprint(total_records)
# pprint(total_null_count)
# pprint(null_ratio_results)

## Metrics to report to prometheus

- These metrics are logged after processing each batch

```
Total records
Records processed
Skipped/ failed Records
Null Count per Field
Schema Drift (Type mismatch)
Constraint Validation (per Field)
```

<pre>
Batch 1   
| --> Partition 1  
|     | --> File 1  
|     |     | --> Record 1  
|     |     |     | --> Field 1  
|     |     |     |     |  
|     |     |     |     |  
|     |     |     |     |  
|     |     |     |     |  
|     |     |     |     Field x  
|     |     |     |  
|     |     |     Record r  
|     |     |  
|     |     File f  
|     |  
|     Partition p  
|  
Batch b  
</pre>

In [80]:
from prometheus_client import Gauge, CollectorRegistry
from time import time

class Metrics:
    schema = ['patients', 'drugs', 'reactions']
    def __init__(self,):
        self.start_time = time()
        self.total_records = {k:0 for k in self.schema}
        self.null_count = {k:{} for k in self.schema}
        self.null_ratio = {k:{} for k in self.schema}

        # Create a custom registry (no collision with global registry)
        self.registry = CollectorRegistry()

        # Define Gauge in this registry
        self.record_gauge = Gauge('total_records', 'Total records per table', ['table'], registry=self.registry)
        self.null_gauge = Gauge('null_count', 'Null count per field', ['table', 'field'], registry=self.registry)
        self.ratio_gauge = Gauge('null_ratio', 'Null ratio per field', ['table', 'field'], registry=self.registry)
        self.processing_time = Gauge('batch_processing_time', 'Time taken to process a batch in seconds', registry=self.registry)

    def update(self, ade):
        """ Update total_records and null_count"""
        patients_df, drugs_df, reactions_df = ade._to_dataframe()

        for s, df in zip(self.schema, [patients_df, drugs_df, reactions_df]):
            # Calculate total records
            self.total_records[s] = self.total_records[s] + df.shape[0]

            # Calculate null counts
            null_count = df.isna().sum().to_dict()
            for k, v in null_count.items():
                self.null_count[s][k] = self.null_count[s].get(k, 0) + v
    
    def _null_ratio(self):
        """ Update null ratio"""
        for s in self.schema:
            for k,v in self.null_count[s].items():
                self.null_ratio[s][k] = float(v / self.total_records[s])

    def publish(self):
        """ Pulish updated metrics to prometheus"""

        # Invoke _null_ratio
        self._null_ratio()

        duration = time() - self.start_time

        # Update Gauge
        for table, count in self.total_records.items():
            self.record_gauge.labels(table=table).set(count)

        for table, fields in self.null_count.items():
            for field, count in fields.items():
                self.null_gauge.labels(table=table, field=field).set(count)

        for table, fields in self.null_ratio.items():
            for field, ratio in fields.items():
                self.ratio_gauge.labels(table=table, field=field).set(ratio)
        
        self.processing_time.set(duration)

        print(self.processing_time._value.get())

In [ ]:
from prometheus_client import REGISTRY, CollectorRegistry

# REGISTRY - Global Registry used by prometheus
# for c in list(REGISTRY._collector_to_names.keys()):
#     REGISTRY.unregister(c)

# Defining a custom Registry
registry = CollectorRegistry()

{}

In [81]:
metrics = Metrics()
for f in os.listdir('tmp'):
    filepath = os.path.join('tmp',f)
    file_json = read_json_file(filepath)

    ade = ADE()
    ade.extractJSON(file_json)
    metrics.update(ade)
    break

metrics.publish()

1.5158696174621582


In [ ]:
# Total records
# Null Count per Field
# Processing time

# Records processed
# Skipped/ failed Records
# Schema Drift
# Constraint Validation (per Field)

In [84]:
!pip freeze | grep "prometheus"

prometheus_client==0.21.1
